# 07 ? Fine-tuned model: with and without RAG

[Notebook guide](README.md) ? [Project overview](../README.md)

Run in a Colab T4 runtime, using the same workflow as notebook 05. This notebook loads **one model once**, then runs two test sections. The four sections across notebooks 07 and 08 share identical questions, references, prompt, 8-bit loading and decoding settings. No training is performed.

The setup uses the simple installation approach from notebooks 05/06, with additional metric packages; it does not repair every preinstalled Colab package. Run installation in a fresh runtime, restart the kernel if requested, then continue with imports. Packages are not pinned to a separately validated environment. The supplied Colab run completed; its answers, metrics, findings and remaining environment limitations are documented in [results/](../results/README.md).

Each section first evaluates the same four isolated questions from notebook 06, then provides the original notebook 05 conversation tests with their original history resets. Quantization stays fixed at 8 bits; its separate effect is not measured. The original strict-context prompt is copied verbatim: **empty context should produce refusal**, which is correct policy following even if reference-answer overlap is low.

For the main fine-tuning comparison, compare section 1 with section 4 on the isolated questions. Their retrieval uses empty history and the same index/settings. Conversation runs retain the original history-aware retriever: model-dependent reformulation can change retrieved passages, so those conversations compare the full pipeline. In the no-RAG branch, history is recorded but is not inserted into the answer prompt, matching the original answer prompt's inputs (`input` and `context`).


## Setup ? install dependencies


In [1]:
%pip install -q langchain langchain-classic langchain-community langchain-huggingface faiss-cpu transformers accelerate sentence-transformers bitsandbytes rouge-score nltk bert-score scikit-learn pandas


### Imports, Drive and model path


In [2]:
import os
import time
import torch
import pandas as pd
from pathlib import Path
from google.colab import drive
from huggingface_hub import login
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline
from langchain_community.vectorstores import FAISS
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline, BitsAndBytesConfig
from langchain_classic.chains.retrieval import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains.history_aware_retriever import create_history_aware_retriever
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder, PromptTemplate
from langchain_core.messages import HumanMessage, AIMessage
from rouge_score import rouge_scorer
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from bert_score import score as bert_score_fn
from sklearn.metrics.pairwise import cosine_similarity
import warnings
import logging
warnings.filterwarnings("ignore")


os.environ['TOKENIZERS_PARALLELISM'] = 'false'


/tmp/ipykernel_8593/3283473732.py:9: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [ ]:
warnings.filterwarnings("ignore")

# Suppress bitsandbytes and autograd warnings
logging.getLogger("bitsandbytes").setLevel(logging.ERROR)
logging.getLogger("bitsandbytes.autograd._functions").setLevel(logging.ERROR)

# Suppress Hugging Face messages
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [4]:
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
db_path = '/content/drive/MyDrive/NLP_proyect/vector_database'
model_id = '/content/drive/MyDrive/NLP_proyect/merged_model'
output_dir = Path('/content/drive/MyDrive/NLP_proyect/evaluation_simple/finetuned')
output_dir.mkdir(parents=True, exist_ok=True)


### Original 8-bit model loading and generation pipeline


In [6]:
tokenizer = AutoTokenizer.from_pretrained(model_id)

quantization_config = BitsAndBytesConfig(
    load_in_8bit=True,
    llm_int8_enable_fp32_cpu_offload=True
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=quantization_config,
    device_map="auto",
    low_cpu_mem_usage=True
)


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

In [7]:
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=512,
    temperature=0.15,
    do_sample=False,
    return_full_text=False,
    repetition_penalty=1.05,
    no_repeat_ngram_size=4,
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.eos_token_id
)
llm = HuggingFacePipeline(pipeline=pipe)


### Retrieval, original prompt and chains


In [8]:
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/paraphrase-multilingual-mpnet-base-v2", model_kwargs={"device": "cpu"})
vector_db = FAISS.load_local(db_path, embedding_model, allow_dangerous_deserialization=True)
retriever = vector_db.as_retriever(search_kwargs={"k": 5})


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

The embedding model runs on CPU to leave T4 memory for the generator. The checkpoint, search depth and default embedding normalization remain the same. Only load the saved pickle index from your own trusted files.


In [9]:
contextualize_q_system_prompt = """Ukoliko postoji istorija razgovora i korisnik postavi pitanje koje se nadovezuje,
reformuliši to pitanje tako da bude samostalno i razumljivo bez istorije razgovora.
NEMOJ odgovarati na pitanje, samo ga reformuliši ako je potrebno, u suprotnom ga vrati identičnog."""

contextualize_q_prompt = ChatPromptTemplate.from_messages([
    ("system", contextualize_q_system_prompt),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])

history_aware_retriever = create_history_aware_retriever(llm, retriever, contextualize_q_prompt)

system_prompt = """<|begin_of_text|><|start_header_id|>system<|end_header_id|>
1. Ti si stručni medicinski asistent i komuniciraš isključivo na srpskom jeziku.
2. Ako u kontekstu nema informacija potrebnih za odgovor, nemoj izmišljati nego jasno reci: "Nažalost, nemam te informacije u svojoj bazi podataka, najbolje se savetuj sa lekarom ili farmaceutom. Informacije sa interneta mogu biti lažne i štetne."
3. Piši svakodnevnim i lako shvatljivim jezikom.
4. Ukoliko je pitanje na latinici, odgovori na latinici. Ukoliko je pitanje na ćirilici, odgovori na ćirilici.
5. Na kraju svakog odgovora OBAVEZNO dodaj: "Pre upotrebe svakog medikamenta, obavezno se posavetuje sa lekarom ili farmaceutom. Dobijeni odgovor nije zamena za stručno mišljenje, već pomoćna alatka."

Pruženi medicinski kontekst:
{context}<|eot_id|><|start_header_id|>user<|end_header_id|>
{input}<|eot_id|><|start_header_id|>assistant<|end_header_id|>"""

qa_prompt = PromptTemplate(
    template=system_prompt,
    input_variables=["context", "input"]
)

question_answer_chain = create_stuff_documents_chain(llm, qa_prompt)
rag_chain = create_retrieval_chain(history_aware_retriever, question_answer_chain)


In [10]:
# The RAG branch is the original chain from notebook 05.
# The other branch uses the same prompt and LLM with an empty context.
no_rag_chain = qa_prompt | llm

def pitaj(pitanje):
    t0 = time.time()
    if koristi_rag:
        rezultat = rag_chain.invoke({'input': pitanje, 'chat_history': chat_history})
        odgovor = rezultat['answer']
        dokumenti = rezultat.get('context', [])
    else:
        odgovor = no_rag_chain.invoke({'input': pitanje, 'context': ''})
        dokumenti = []
    trajanje = time.time() - t0
    chat_history.append(HumanMessage(content=pitanje))
    chat_history.append(AIMessage(content=odgovor))
    print(f'Q: {pitanje}\nA: {odgovor}\n[{trajanje:.1f}s]\n')
    return odgovor, dokumenti, trajanje


### Shared reference cases and metric definitions


Questions, reference strings (including existing typos), and metric definitions are copied from notebook 06. This four-question sample is a pilot. The original ROUGE tokenizer has limitations for Cyrillic; interpret those scores accordingly. Metrics measure reference similarity and do not establish medical correctness. BERTScore runs on CPU, avoiding a second transformer on the T4.


In [11]:
# TODO: populate referentni_odgovor with a reference answer from the knowledge base
# (a short, factually correct answer expected from the model).

eval_dataset_odgovori = [
    {
        "pitanje": "Kada ne smem da uzimam lek Dovicin?",
        "referentni_odgovor": '''Lek Dovicin ne smete uzimati:
- ukoliko ste alergični (preosetljivi) na doksiciklin ili neki drugi lek iz grupe tetraciklina ili na bilo koju od od pomoćnih supstanci ovog leka (navedene u odeljku 6);
- ukoliko ste trudni ili planirate trudnoću;
- ukoliko dojite;
- ukoliko imate oboljenje praćeno otežanim gutanjem.
Lek Dovicin ne treba koristiti tokom perioda razvoja zuba (u periodu trudnoće, kod odojčadi i dece mlađe od 8 godina)
''',
    },
    {
        "pitanje": "Koja je uobičajena doza za lek Nimulid?",
        "referentni_odgovor": "Lek Nimulid uvek uzimajte tačno onako kako Vam je objasnio Vaš lekar. Ukoliko niste sigurni, proverite sa Vašim lekarom ili farmaceutom. Uobičajena doza leka je jedna tableta (100 mg) dva puta dnevno, posle obroka.",
    },
    {
        "pitanje": "Da li deca smeju da konzumiraju Pressing?",
        "referentni_odgovor": "Deca: Nemojre davati lel Pressing sirup deci mlađoj od 2 godine.",
    },
    {
        "pitanje": "Za šta se primenjuje lek Zoloft",
        "referentni_odgovor": '''Lek Zoloft se može primenjivati za lečenje:
• Depresije i prevenciju ponovnog javljanja depresije (kod odraslih).
• Socijalnog anksioznog poremećaja (kod odraslih).
• Posttraumatskog stresnog poremećaja (kod odraslih).
• Paničnog poremećaja (kod odraslih).
• Opsesivno-kompulzivnog poremećaja (kod odraslih i dece i adolescenata uzrasta od 6 do 17 godina).''',
    },

]

len(eval_dataset_odgovori)


4

In [12]:
rouge_l_scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=False)
smoothing = SmoothingFunction().method1


def racunaj_rouge_l(generisan, referentan):
    rezultat = rouge_l_scorer.score(referentan, generisan)
    return rezultat["rougeL"].fmeasure


def racunaj_bleu(generisan, referentan):
    referenca_tok = [referentan.split()]
    kandidat_tok = generisan.split()
    return sentence_bleu(referenca_tok, kandidat_tok, smoothing_function=smoothing)


def racunaj_cos_similarity(generisan, referentan):
    emb_gen = embedding_model.embed_query(generisan)
    emb_ref = embedding_model.embed_query(referentan)
    sim = cosine_similarity([emb_gen], [emb_ref])[0][0]
    return float(sim)


## Test 1 ? Fine-tuned model with RAG

Run this section from the first cell. The test only changes the selected chain; the model, prompt and decoding settings are shared with the other section. Four isolated questions use empty history individually. Generated answers, retrieved text and metrics are saved as CSV files on Drive, overwriting earlier files for this section when rerun. Save the notebook to retain conversation outputs too.


In [13]:
koristi_rag = True
test_name = 'test_1_finetuned_rag'
chat_history = []


In [14]:
# Same four independent questions in all four tests.
generisani_odgovori = []
redovi = []
for primer in eval_dataset_odgovori:
    chat_history = []
    odgovor, dokumenti, trajanje = pitaj(primer['pitanje'])
    generisani_odgovori.append(odgovor)
    redovi.append({
        'Pitanje': primer['pitanje'],
        'Odgovor': odgovor,
        'Referentni odgovor': primer['referentni_odgovor'],
        'Vreme (s)': trajanje,
        'Izvori': '; '.join(str(d.metadata.get('izvor_fajl', '')) for d in dokumenti),
        'Kontekst': '\n\n'.join(d.page_content for d in dokumenti),
    })
# Save answers before calculating metrics, so a metric error does not lose them.
df_odgovori = pd.DataFrame(redovi)
df_odgovori.to_csv(output_dir / f'{test_name}_answers.csv', index=False)
display(df_odgovori[['Pitanje', 'Odgovor', 'Vreme (s)', 'Izvori']])


[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Q: Kada ne smem da uzimam lek Dovicin?
A: 

Ne smete uzimat' lek Dovičin ukoliko:

- ste alergičan (preosvetljiv) na dokičiklin ili nekli drugi leik iz grupe tetraciklina, ili na bili koju od pomoći supstancija ovog leika (navedena u odelju 6);
- ste trudne ili planirate roditeljstvo;
- dojite;
- imate obolenje praćenim otežnim gutanjem.

Pre upotrebu svakog leka obavezno saovetujte se sa lekarim ili farmaceutom.
[29.3s]

Q: Koja je uobičajena doza za lek Nimulid?
A: 

Uobičajan doza lek Nimuld je jedna tablete (100 mg), dve puta dnevono, posle jela.
[8.2s]

Q: Da li deca smeju da konzumiraju Pressing?
A: 

Nažalosti, nemam informaciju o dozama i upotrebi leka Pressing kod dece. Najbolje je da se savetujete sa lekarjem ili farmaceutima pre upotrebe bilo kog leka, posebno ako je pacijenat deca.
[11.3s]

Q: Za šta se primenjuje lek Zoloft
A: 

Lek Zolof se može koristiti za lečenju sledećih stanja:

- Depresija i prevencija ponovnoga javljanje depresiji kod odrasle populacije.
- Socijaln

,Pitanje,Odgovor,Vreme (s),Izvori
0,Kada ne smem da uzimam lek Dovicin?,\n\nNe smete uzimat' lek Dovičin ukoliko:\n\n-...,29.316026,000020576 2024 59010 007 000 515 002 04 001.pd...
1,Koja je uobičajena doza za lek Nimulid?,\n\nUobičajan doza lek Nimuld je jedna tablete...,8.185271,515-01-05090-22-001.pdf; 515-01-05090-22-001.p...
2,Da li deca smeju da konzumiraju Pressing?,"\n\nNažalosti, nemam informaciju o dozama i up...",11.322581,000457297 2023 59010 007 000 515 021 04 001.pd...
3,Za šta se primenjuje lek Zoloft,\n\nLek Zolof se može koristiti za lečenju sle...,24.555225,515-01-00535-22-001.pdf; 515-01-00535-22-001.p...


In [15]:
referentni_odgovori = [p['referentni_odgovor'] for p in eval_dataset_odgovori]
bert_p, bert_r, bert_f1 = bert_score_fn(
    generisani_odgovori, referentni_odgovori,
    lang='sr', device='cpu', batch_size=1, verbose=False,
)
rezultati_odgovora = []
for i, primer in enumerate(eval_dataset_odgovori):
    gen = generisani_odgovori[i]
    ref = primer['referentni_odgovor']
    rezultati_odgovora.append({
        'Pitanje': primer['pitanje'],
        'ROUGE-L': racunaj_rouge_l(gen, ref),
        'BLEU': racunaj_bleu(gen, ref),
        'BERTScore (F1)': bert_f1[i].item(),
        'Cosine similarity': racunaj_cos_similarity(gen, ref),
        'Vreme (s)': redovi[i]['Vreme (s)'],
    })
df_metrike = pd.DataFrame(rezultati_odgovora)
display(df_metrike)
display(df_metrike.drop(columns='Pitanje').mean().to_frame('PROSEK'))
df_metrike.to_csv(output_dir / f'{test_name}_metrics.csv', index=False)


config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  714MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


,Pitanje,ROUGE-L,BLEU,BERTScore (F1),Cosine similarity,Vreme (s)
0,Kada ne smem da uzimam lek Dovicin?,0.406015,0.016033,0.818883,0.945223,29.316026
1,Koja je uobičajena doza za lek Nimulid?,0.307692,0.009527,0.788324,0.848245,8.185271
2,Da li deca smeju da konzumiraju Pressing?,0.046512,0.006244,0.636401,0.654273,11.322581
3,Za šta se primenjuje lek Zoloft,0.330097,0.013319,0.781340,0.963965,24.555225


,PROSEK
ROUGE-L,0.272579
BLEU,0.011281
BERTScore (F1),0.756237
Cosine similarity,0.852926
Vreme (s),18.344776


### Optional ? original conversation tests from notebook 05

These calls and the two history resets are copied in the original order. They are separate from the four isolated questions above. Run the following cells only if you also want to inspect the original multi-turn examples; they do not contribute to the metric table.


##### Session A: Dovicin in Latin script

**History reset:** the next cell starts a new conversation.

In [16]:
chat_history = []


In [17]:
pitaj("Kada ne smem da uzimam lek Dovicin?")

Q: Kada ne smem da uzimam lek Dovicin?
A: 

Ne smete uzimat' lek Dovičin ukoliko:

- ste alergičan (preosvetljiv) na dokičiklin ili nekli drugi leik iz grupe tetraciklina, ili na bili koju od pomoći supstancija ovog leika (navedena u odelju 6);
- ste trudne ili planirate roditeljstvo;
- dojite;
- imate obolenje praćenim otežnim gutanjem.

Pre upotrebu svakog leka obavezno saovetujte se sa lekarim ili farmaceutom.
[27.6s]



("\n\nNe smete uzimat' lek Dovičin ukoliko:\n\n- ste alergičan (preosvetljiv) na dokičiklin ili nekli drugi leik iz grupe tetraciklina, ili na bili koju od pomoći supstancija ovog leika (navedena u odelju 6);\n- ste trudne ili planirate roditeljstvo;\n- dojite;\n- imate obolenje praćenim otežnim gutanjem.\n\nPre upotrebu svakog leka obavezno saovetujte se sa lekarim ili farmaceutom.",
 [Document(id='ad3c9945-5812-441f-a91b-736276418bd1', metadata={'Naslov_1': '**Ako ste zaboravili da uzmete lek Diklofen DUO**', 'izvor_fajl': '000020576 2024 59010 007 000 515 002 04 001.pdf'}, page_content='Ako ste zaboravili da uzmete lek Diklofen DUO\nAko je prošlo malo više od 12 sati od poslednje doze leka, uzmite propuštenu dozu i sačekajte 8 do 12 sati do sledeće doze. Ne uzimajte duplu dozu da biste nadoknadili propuštenu dozu. Ne smete uzeti više od dve doze tokom 24 sata.  \nAko imate dodatnih pitanja o upotrebi ovog leka, obratite se svom lekaru ili farmaceutu.  \n4 od 7'),
  Document(id='6961

In [18]:
pitaj("A šta ako uzimam i drugi lek pored Dovicina? Šta da radim?")

Q: A šta ako uzimam i drugi lek pored Dovicina? Šta da radim?
A: 

Ako uzimate i drugi lijek pored Dovicina, preporučuje se da obavijestite svoga liječnika ili farmaceutu. Lekovi mogu uticati jedni na druge, što može uticati na njihovo dejstvo ili povećavati rizik od nepoželjnih reakcija.
[109.6s]



('\n\nAko uzimate i drugi lijek pored Dovicina, preporučuje se da obavijestite svoga liječnika ili farmaceutu. Lekovi mogu uticati jedni na druge, što može uticati na njihovo dejstvo ili povećavati rizik od nepoželjnih reakcija.',
 [Document(id='4eebb4e3-150d-4389-8d29-01dabbdac32a', metadata={'Naslov_1': '**Drugi lekovi i lek Dovicin**', 'izvor_fajl': '515-01-00434-19-001.pdf'}, page_content='Drugi lekovi i lek Dovicin\nObavestite Vašeg lekara ili farmaceuta ukoliko uzimate ili ste donedavno uzimali ili ćete možda uzimati bilo koje druge lekove. uključujući i lekove koji se mogu nabaviti bez lekarskog recepta.  \nSledeći lekovi mogu da smanje efikasnost leka Dovicin:  \n- antacidi (lekovi koji se koriste kod poremećaja varenja), lekovi koji sadrže gvožđe, cink ili bizmut ukoliko se uzimaju oralno. Ove lekove ne treba uzimati u isto doba dana kada se uzima i lek Dovicin;  \n- karbamazepin, fenitoin (lekovi koji se koriste u terapiji epilepsije) i barbiturati (koji se koriste u terapiji

#### Session A continued: Dovicin in Cyrillic

The original notebook does not reset history here. Earlier turns remain available.

In [19]:
pitaj("Када не смем да узимам лек Dovicin?")

Q: Када не смем да узимам лек Dovicin?
A: 

Lek Dovičin ne smete da uzimate ukoliko:

- ste alerhici (preosеtlijivi) na doksiciklin или на било коју од помоћних супстанци овог лека (наведене у одељку 6);
- уколико сте трудни или планирате трудноћу;
- уколко дојите;
- уколо имате оболење праћено отеженим гутањем.
[51.9s]



('\n\nLek Dovičin ne smete da uzimate ukoliko:\n\n- ste alerhici (preosеtlijivi) na doksiciklin или на било коју од помоћних супстанци овог лека (наведене у одељку 6);\n- уколико сте трудни или планирате трудноћу;\n- уколко дојите;\n- уколо имате оболење праћено отеженим гутањем.',
 [Document(id='9a186257-8345-435d-a795-caed900538a6', metadata={'Naslov_1': '**Lek Dovicin ne smete uzimati:**', 'izvor_fajl': '515-01-00434-19-001.pdf'}, page_content='Lek Dovicin ne smete uzimati:\n- ukoliko ste alergični (preosetljivi) na doksiciklin ili neki drugi lek iz grupe tetraciklina ili na bilo koju od  \n- od pomoćnih supstanci ovog leka (navedene u odeljku 6);  \n- ukoliko ste trudni ili planirate trudnoću;  \n- ukoliko dojite;  \n- ukoliko imate oboljenje praćeno otežanim gutanjem.  \nLek Dovicin ne treba koristiti tokom perioda razvoja zuba (u periodu trudnoće, kod odojčadi i dece mlađe od 8 godina), jer njegova primena može dovesti do trajne promene boje zuba (promena boje zuba u žućkastosivk

In [20]:
pitaj("Зашто си ми генерисао одговор на латиници?")

[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Q: Зашто си ми генерисао одговор на латиници?
A: 
Поштовани корисник, извините ми да сам генерисовао одговор у латиницама. Као АИ, моја основна функција је пружање информација и помоћи корисницима, али не могу да контролишем формат у коме се одговори објављују. Међутим, могу да промените формат одговора да одговара вашим захтевима. Ако желите да одговор буде у српском језику, могу да га претворим у србски. Можете ми рећи да ли желите да се одговора претвори у србијански и ћу радити на томе.
[62.7s]



('\nПоштовани корисник, извините ми да сам генерисовао одговор у латиницама. Као АИ, моја основна функција је пружање информација и помоћи корисницима, али не могу да контролишем формат у коме се одговори објављују. Међутим, могу да промените формат одговора да одговара вашим захтевима. Ако желите да одговор буде у српском језику, могу да га претворим у србски. Можете ми рећи да ли желите да се одговора претвори у србијански и ћу радити на томе.',
 [Document(id='4eebb4e3-150d-4389-8d29-01dabbdac32a', metadata={'Naslov_1': '**Drugi lekovi i lek Dovicin**', 'izvor_fajl': '515-01-00434-19-001.pdf'}, page_content='Drugi lekovi i lek Dovicin\nObavestite Vašeg lekara ili farmaceuta ukoliko uzimate ili ste donedavno uzimali ili ćete možda uzimati bilo koje druge lekove. uključujući i lekove koji se mogu nabaviti bez lekarskog recepta.  \nSledeći lekovi mogu da smanje efikasnost leka Dovicin:  \n- antacidi (lekovi koji se koriste kod poremećaja varenja), lekovi koji sadrže gvožđe, cink ili biz

In [21]:
pitaj("Али већ си генерисао претходни одговор на српском. Генериши одговор на ћирилици.")

Q: Али већ си генерисао претходни одговор на српском. Генериши одговор на ћирилици.
A: 
Молим вас да поново поставите питање на србијанци.
[89.4s]



('\nМолим вас да поново поставите питање на србијанци.',
 [Document(id='8b5bf5ff-b39f-4451-9c61-a32145cfd551', metadata={'izvor_fajl': '000457796 2023 59010 007 000 515 021 04 001.pdf'}, page_content='**_<u>UPUTSTVO ZA LEK</u>_**'),
  Document(id='37155948-d356-447b-9a70-50f0bd3af3aa', metadata={'izvor_fajl': '515-01-02931-21-001.pdf'}, page_content='**<u>UPUTSTVO ZA LEK</u>**'),
  Document(id='b434c452-06b9-4f36-9a16-3363ad1c7706', metadata={'izvor_fajl': '515-01-00448-23-001.pdf'}, page_content='**<u>UPUTSTVO ZA LEK</u>**'),
  Document(id='264ac4b0-e50d-471f-84f3-90c21441b50a', metadata={'izvor_fajl': '515-01-01364-21-001.pdf'}, page_content='**<u>UPUTSTVO ZA LEK</u>**'),
  Document(id='fdeb18fe-6ec7-4d3a-8829-85ab6661d448', metadata={'izvor_fajl': '000457379 2023 59010 007 000 515 021 04 001.pdf'}, page_content='**<u>UPUTSTVO ZA LEK</u>**')],
 89.37341403961182)

#### Session A continued: Nimulid

This block continues the same history; it is not an isolated test.

In [22]:
pitaj("Која је уобичајена доза за лек Nimulid?")

Q: Која је уобичајена доза за лек Nimulid?
A: 
Уобићајена дозa лека Нимулид је једна таблета (100 мг) два пута дневно, послије оброка.
[32.5s]



('\nУобићајена дозa лека Нимулид је једна таблета (100 мг) два пута дневно, послије оброка.',
 [Document(id='18dc6613-2c47-4baa-87af-be9f59850bd5', metadata={'Naslov_1': '**3. Kako se uzima lek Nimulid**', 'izvor_fajl': '515-01-05090-22-001.pdf'}, page_content='3. Kako se uzima lek Nimulid\nLek Nimulid uvek uzimajte tačno onako kako Vam je objasnio Vaš lekar. Ukoliko niste sigurni, proverite sa Vašim lekarom ili farmaceutom.  \nUobičajena doza leka je jedna tableta (100 mg) dva puta dnevno, posle obroka.  \nKoristite lek Nimulid u što je moguće kraćem vremenskom periodu i ne duže od 15 dana tokom jednog terapijskog ciklusa.  \nNeželjena dejstva se mogu smanjiti korišćenjem minimalne efektivne doze koja dovodi do kontrole simptoma u najkraćem mogućem vremenskom periodu.'),
  Document(id='56319634-d504-412b-a881-2ee625c87919', metadata={'Naslov_1': '<u>Način primene</u>', 'izvor_fajl': '515-01-04818-21-001.pdf'}, page_content='Način primene\nObložene tablete treba uzimati sa dovoljnom ko

In [23]:
pitaj("Мислим да сам случајно прекорачио дозу. Узео сам по једну таблету више сваки пут. Шта ми се може десити?")

Q: Мислим да сам случајно прекорачио дозу. Узео сам по једну таблету више сваки пут. Шта ми се може десити?
A: 

Ако сте случајно преузели више лека Нимулид од предвиђене дозе, најбоље је да се обратите свом лекару или фармацеуту. Они могу да вас упуте како да се понашајте и да ли ће бити потребна медицинска помоћ.
[111.2s]



('\n\nАко сте случајно преузели више лека Нимулид од предвиђене дозе, најбоље је да се обратите свом лекару или фармацеуту. Они могу да вас упуте како да се понашајте и да ли ће бити потребна медицинска помоћ.',
 [Document(id='fa5e0735-b385-4c56-aaac-973054190cee', metadata={'Naslov_1': '**Upozorenja i mere opreza**', 'izvor_fajl': '515-01-05090-22-001.pdf'}, page_content='Upozorenja i mere opreza\nRazgovarajte sa svojim lekarom ili farmaceutom pre nego što uzmete lek Nimulid.  \nLek Nimulid može da poveća rizik za nastanak srčanog udara (infarkt miokarda) ili moždanog udara, naročito ukoliko se koriste velike doze i ako se lek primenjuje duže od preporučenog trajanja terapije.  \nAko imate srčane probleme, ako ste imali srčani udar ili ako mislite da imate faktore rizika (npr. visok krvni pritisak, dijabetes, povećane vrednosti holesterola ili ako pušite) koji su u vezi sa ovim bolestima, molimo Vas da o tome obavestite svog lekara.  \nU slučajevima pojave ozbiljnih alergijskih reakci

In [24]:
pitaj("Такође сам конзумирао алкохол. Да ли ми се нешто може десити?")

Q: Такође сам конзумирао алкохол. Да ли ми се нешто може десити?
A: 
Да, конзумовањем алкохола са леком Нимулидом може доћи до повећањег ризика од нежељених ефеката. Алкохол и Нимулид могу да се интерактирају и потенцијално узрокују побољшање ризика од неких нежељних ефеката, укључујући: - Повећани ризик од алергиjsких реакцији - Повеćани ризик отмицања функције бубрега - Повечани ризик оштећењa функције жлезда - Повеђани ризик смаѣњеђењe функције циркулации - Повеќани ризик повећаног крвног притиска - Повецање риска од срчаних проблема, укљућујуци инфаркт миокарда и можданог удара.
[76.0s]



('\nДа, конзумовањем алкохола са леком Нимулидом може доћи до повећањег ризика од нежељених ефеката. Алкохол и Нимулид могу да се интерактирају и потенцијално узрокују побољшање ризика од неких нежељних ефеката, укључујући: - Повећани ризик од алергиjsких реакцији - Повеćани ризик отмицања функције бубрега - Повечани ризик оштећењa функције жлезда - Повеђани ризик смаѣњеђењe функције циркулации - Повеќани ризик повећаног крвног притиска - Повецање риска од срчаних проблема, укљућујуци инфаркт миокарда и можданог удара.',
 [Document(id='fa5e0735-b385-4c56-aaac-973054190cee', metadata={'Naslov_1': '**Upozorenja i mere opreza**', 'izvor_fajl': '515-01-05090-22-001.pdf'}, page_content='Upozorenja i mere opreza\nRazgovarajte sa svojim lekarom ili farmaceutom pre nego što uzmete lek Nimulid.  \nLek Nimulid može da poveća rizik za nastanak srčanog udara (infarkt miokarda) ili moždanog udara, naročito ukoliko se koriste velike doze i ako se lek primenjuje duže od preporučenog trajanja terapije

##### Session B: Pressing

**History reset:** the next cell starts a new conversation.

In [25]:
chat_history = []


In [26]:
pitaj("Da li deca smeju da konzumiraju Pressing?")

Q: Da li deca smeju da konzumiraju Pressing?
A: 

Nažalosti, nemam informaciju o dozama i upotrebi leka Pressing kod dece. Najbolje je da se savetujete sa lekarjem ili farmaceutima pre upotrebe bilo kog leka, posebno ako je pacijenat deca.
[11.1s]



('\n\nNažalosti, nemam informaciju o dozama i upotrebi leka Pressing kod dece. Najbolje je da se savetujete sa lekarjem ili farmaceutima pre upotrebe bilo kog leka, posebno ako je pacijenat deca.',
 [Document(id='e3ccf126-4ed7-4502-8739-81fc19738569', metadata={'Naslov_1': '_<u>Deca</u>_', 'izvor_fajl': '000457297 2023 59010 007 000 515 021 04 001.pdf'}, page_content='Deca\nLek nije namenjen deci uzrasta ispod 12 godina. Za mlađu decu dostupne su pogodnije formulacije u obliku sirupa.'),
  Document(id='17fc70fb-0f25-4f89-b33d-12203264496a', metadata={'Naslov_1': '**Upotreba kod dece i adolescenata**', 'izvor_fajl': '515-01-05165-21-001.pdf'}, page_content='Upotreba kod dece i adolescenata\nDeca mlađa od 10 godina ne smeju da uzimaju laksative (lekove za pražnjenje creva) bez saveta lekara. Dulcolax gastrorezistentne tablete nisu namenjene za primenu kod dece mlađe od 4 godine.'),
  Document(id='1c878081-b474-42b2-8fc8-afcd6fd77627', metadata={'Naslov_1': '**Primena kod dece i adolescen

In [27]:
pitaj("Da li odrasle osobe mogu da konzumiraju Pressing?")

Q: Da li odrasle osobe mogu da konzumiraju Pressing?
A: 
Da, odrasli mogu da konsumiraju Pressin sirup, ali je potrebno pažljivo pratiti doziranje i preporuke.
[19.2s]



('\nDa, odrasli mogu da konsumiraju Pressin sirup, ali je potrebno pažljivo pratiti doziranje i preporuke.',
 [Document(id='90de289d-000f-41a0-9135-7898f2097ecb', metadata={'Naslov_1': '**Следеће информације су намењене искључиво здравственим радницима**', 'Naslov_2': '_Дозирање_', 'izvor_fajl': '515-01-03530-21-001.pdf'}, page_content='Следеће информације су намењене искључиво здравственим радницима - Дозирање\nДозирање се одређује према интензитету болa или вредностима телесне температуре и индивидуалном одговору на лек ANALGIN. Неопходно је изабрати најнижу дозу којом се контролишу бол и повишена телесна температура.  \nДеци и адолесцентима узраста до 14 година, метамизол у дози 8 – 16 mg по kg телесне масе може се давати као једна доза. У случају повишене телесне температуре, за децу је по правилу довољна доза од 10 mg метамизола по килограму телесне масе. Одрасли и адолесценти од 15 година или старији (> 53 kg телесне масе) могу узимати до 1000 mg као једну дозу.  \nЗависно од мак

In [28]:
pitaj("Ali ja te ne pitam za lek Crestora, već za lek Pressing?")

Q: Ali ja te ne pitam za lek Crestora, već za lek Pressing?
A: 

Žao mi je što sam pogrešno razumeo vaš pitanje. Da li biste mogli da ponovite pitanje o leku Pressing? Da li je to isti lek ili drugi?
[42.4s]



('\n\nŽao mi je što sam pogrešno razumeo vaš pitanje. Da li biste mogli da ponovite pitanje o leku Pressing? Da li je to isti lek ili drugi?',
 [Document(id='b8c50681-80a5-4bcd-87c6-eca06a6a7132', metadata={'Naslov_2': '**Upozorenja i mere opreza**', 'izvor_fajl': '002132821 2024 59010 007 000 515 021 04 001.pdf'}, page_content='Upozorenja i mere opreza\nRazgovarajte sa svojim lekarom ili farmaceutom pre nego što uzmete lek Crestor:  \n- **ukoliko imate probleme sa bubrezima,**  \n- **ukoliko imate probleme sa jetrom,**  \n- **ukoliko ste imali bilo kakve učestale ili neobjašnjive bolove u mišićima** , ako ste Vi ili neko od članova Vaše porodice ranije imali problema sa mišićima, ili ukoliko ste ranije imali problema sa mišićima kada ste uzimali druge lekove za smanjenje vrednosti holesterola. Odmah obavestite svog lekara ukoliko imate neobjašnjive bolove u mišićima, a posebno ako se ne osećate dobro ili imate povišenu telesnu temperaturu. Takođe, recite svom lekaru ili farmaceutu uko

#### Session B continued: more specific wording

History is retained while the wording of the question changes.

In [29]:
pitaj("Da li deca smeju da konzumiraju Pressing sirup?")

Q: Da li deca smeju da konzumiraju Pressing sirup?
A: 

Da li dečaci mogu da konzumiraše Pressing sirap? Da li je Pressing sirip za decu?
[31.7s]



('\n\nDa li dečaci mogu da konzumiraše Pressing sirap? Da li je Pressing sirip za decu?',
 [Document(id='293b7beb-51ff-4343-a94e-5632781f8214', metadata={'Naslov_1': '**1. Šta je lek Pressing sirup i čemu je namenjen**', 'izvor_fajl': '000456529 2023 59010 007 000 515 021 04 001.pdf'}, page_content='1. Šta je lek Pressing sirup i čemu je namenjen\nLek Pressing sirup sadrži aktivnu supstancu loratadain koja prpada grupi lekova poznatih pod nazivom antihistaminici (blokatori H1 receptora). Antihistaminici pomažu u olakšanju alergijskih simptoma sprečavajući efekte supstance pod nazivom histamin, koja se stvara u organizmu.  \nLek Pressing sirup se koristi za ublažavanje simptoma povezanih sa alergijskim rinitisom (npr. polenska kijavica), koju karakteriše kijanje, curenje iz nosa, svrab i osećaj nadraženosti u nosu, peckanje ili svrab u očima kod odraslih i dece starije od 2 godine.  \nLek Pressing sirup se takođe može upotrebiti za ublažavanje simptoma hronične idiopatske urtikarije (ko

In [30]:
pitaj("Šta je to Pressing sirup?")

Q: Šta je to Pressing sirup?
A: 

Pressing sirup je lek koji sadrži paracetamol kao glavnu aktivnu supstavu. Paracetamol je analgetik i antipiretik, što znači da služi za ublaživanje bolova i smanjenje temperature.
[91.0s]



('\n\nPressing sirup je lek koji sadrži paracetamol kao glavnu aktivnu supstavu. Paracetamol je analgetik i antipiretik, što znači da služi za ublaživanje bolova i smanjenje temperature.',
 [Document(id='921487eb-b677-4629-9901-b67836a6d106', metadata={'Naslov_1': '**Režim izdavanja leka:**', 'izvor_fajl': '515-01-00519-21-002.pdf'}, page_content='Režim izdavanja leka:\nLek se može izdavati uz lekarski recept.  \n9 od 10  \n515-01-00519-21-002  od    11.01.2022.'),
  Document(id='44b3b689-67da-4a04-9e6d-843f1a8863da', metadata={'Naslov_1': '**Pažljivo pročitajte ovo uputstvo, pre nego što počnete da primate ovaj lek, jer ono sadrži informacije koje su važne za Vas.**', 'izvor_fajl': '4.pdf'}, page_content='Pažljivo pročitajte ovo uputstvo, pre nego što počnete da primate ovaj lek, jer ono sadrži informacije koje su važne za Vas.\n- Uputstvo sačuvajte. Može biti potrebno da ga ponovo pročitate.  \n- Ako imate dodatnih pitanja, obratite se svom lekaru, farmaceutu ili medicinskoj sestri. 

In [31]:
pitaj("Da li deca smeju da konzumiraju Pressing sirup?")

Q: Da li deca smeju da konzumiraju Pressing sirup?
A: 

Deca mlaђa od 2 године ne smeju da uzimaju lek Pressin sirup.
[20.2s]



('\n\nDeca mlaђa od 2 године ne smeju da uzimaju lek Pressin sirup.',
 [Document(id='cb0fbcb0-5b85-4586-90f6-5da306066857', metadata={'Naslov_1': '**Upozorenja i mere opreza**', 'izvor_fajl': '000456529 2023 59010 007 000 515 021 04 001.pdf'}, page_content='Upozorenja i mere opreza\nRazgovarajte sa Vašim lekarom ili farmaceutom pre nego što uzmete lek Pressing sirup:  \n- Ukoliko imate teško oboljenje jetre  \n- Ukoliko treba da uradite bilo kakve kožne alergijske testove, nemojte uzimati lek Pressing sirup najmanje dva dana pre izvođenja tih testova, jer ovaj lek može imati uticaja na rezultate testova.  \nDeca:  \nNemojre davati lel Pressing sirup deci mlađoj od 2 godine.'),
  Document(id='1c878081-b474-42b2-8fc8-afcd6fd77627', metadata={'Naslov_1': '**Primena kod dece i adolescenata**', 'izvor_fajl': '515-01-05165-21-001.pdf'}, page_content='Primena kod dece i adolescenata\nLek Dulcolax ne sme se primenjivati kod dece mlađe od 4 godine. Deca mlađa od 10 godina smeju se lečiti isklju

In [32]:
pitaj("A deca starija od 2 godine?")

Q: A deca starija od 2 godine?
A: 

Ukoliko je pacient stariji od 2, lek Cresto se ne sme koristiti deca mlađa od 6.
[42.1s]



('\n\nUkoliko je pacient stariji od 2, lek Cresto se ne sme koristiti deca mlađa od 6.',
 [Document(id='90de289d-000f-41a0-9135-7898f2097ecb', metadata={'Naslov_1': '**Следеће информације су намењене искључиво здравственим радницима**', 'Naslov_2': '_Дозирање_', 'izvor_fajl': '515-01-03530-21-001.pdf'}, page_content='Следеће информације су намењене искључиво здравственим радницима - Дозирање\nДозирање се одређује према интензитету болa или вредностима телесне температуре и индивидуалном одговору на лек ANALGIN. Неопходно је изабрати најнижу дозу којом се контролишу бол и повишена телесна температура.  \nДеци и адолесцентима узраста до 14 година, метамизол у дози 8 – 16 mg по kg телесне масе може се давати као једна доза. У случају повишене телесне температуре, за децу је по правилу довољна доза од 10 mg метамизола по килограму телесне масе. Одрасли и адолесценти од 15 година или старији (> 53 kg телесне масе) могу узимати до 1000 mg као једну дозу.  \nЗависно од максималне дневне дозе,

#### Session B continued: repeated standalone wording

Although the question is self-contained, the original code does not reset conversation history.

In [33]:
pitaj("Šta je to Pressing sirup?")

Q: Šta je to Pressing sirup?
A: 

Pressing sirup je lek koji sadrži loratadin kao aktivnu supstan-cu. Loratadin je antihistaminski lek koji se koristi u liječenju alergija, kao što su alergiјa na polen, a takođe se može koristiti za ublaživanje simptoma kao što su svrab, curenja iz nosa i peckanje u očima.
[103.4s]



('\n\nPressing sirup je lek koji sadrži loratadin kao aktivnu supstan-cu. Loratadin je antihistaminski lek koji se koristi u liječenju alergija, kao što su alergiјa na polen, a takođe se može koristiti za ublaživanje simptoma kao što su svrab, curenja iz nosa i peckanje u očima.',
 [Document(id='7cdd3365-ac7d-465e-b657-e4169b757408', metadata={'Naslov_2': '**Deca i adoslescenti**', 'izvor_fajl': '002132821 2024 59010 007 000 515 021 04 001.pdf'}, page_content='Deca i adoslescenti\n- **ukoliko je pacijent mlađi od 6 godina:** lek Crestor se ne sme primenjivati deci mlađoj od 6 godina,  \n- **ukoliko je pacijent mlađi od 18 godina** : lek Crestor u dozi od 40 mg nije pogodan za primenu kod dece i adolescenata mlađih od 18 godina.'),
  Document(id='aefca475-56f5-4929-ac40-bb269560c6c2', metadata={'Naslov_1': '_<u>Pedijatrijska populacija</u>_', 'izvor_fajl': '000456529 2023 59010 007 000 515 021 04 001.pdf'}, page_content='Pedijatrijska populacija\n_Kod dece od 2 do 12 godina_ **_doziranj

#### Session B continued: Zoloft

The existing history remains in use.

In [34]:
pitaj("Za šta se primenjuje lek Zoloft")

Q: Za šta se primenjuje lek Zoloft
A: 

Lek Zolof se može koristiti za lečenju sledećih stanja:

1. Depresija i prevencija ponovnoga javljanje depresiji kod odrasle populacije.
2. Socijalna anksiozanost kod odrasnih osoba.
3. Posttraumatični stresni sindrom kod odraslaca.
4. Panični sindrom u odrasloj populaciji.
5. Opsesivnokompulzivan sindrom kod zdravih odraslog stanovništva i dece i adolescenta starosne grupe od 6-17 godina.
[53.1s]



('\n\nLek Zolof se može koristiti za lečenju sledećih stanja:\n\n1. Depresija i prevencija ponovnoga javljanje depresiji kod odrasle populacije.\n2. Socijalna anksiozanost kod odrasnih osoba.\n3. Posttraumatični stresni sindrom kod odraslaca.\n4. Panični sindrom u odrasloj populaciji.\n5. Opsesivnokompulzivan sindrom kod zdravih odraslog stanovništva i dece i adolescenta starosne grupe od 6-17 godina.',
 [Document(id='f283d2da-8dbb-4e70-af87-c1c9aabc81fe', metadata={'Naslov_1': '**Upozorenja i mere opreza**', 'izvor_fajl': '515-01-00535-22-001.pdf'}, page_content='Upozorenja i mere opreza\nRazgovarajte sa svojim lekarom ili farmaceutom pre nego što uzmete lek Zoloft.  \nLekovi ne odgovaraju uvek svakome. Pre nego što uzmete lek Zoloft, molimo Vas recite Vašem lekaru da li imate ili ste nekada imali bilo koje od navedenih stanja:  \n- Ukoliko imate epilepsiju ili napade u anamnezi. Ukoliko imate napad, odmah kontaktirajte Vašeg lekara.  \n- Ukoliko ste bolovali od manično-depresivne bol

In [35]:
pitaj("Započeo sam terapiju ovim lekom, da li je normalno što ponekad pomislim o samopovređivanju?")

Q: Započeo sam terapiju ovim lekom, da li je normalno što ponekad pomislim o samopovređivanju?
A: 

Nažalosti, nemam informacije o tome da li je pomenuto ponašanje normalno tokom terapie Spalmotila. Pre upotrebe bilo kog leka obavezno je da se posaveti sa lekarim ili farmaceutom.
[26.1s]



('\n\nNažalosti, nemam informacije o tome da li je pomenuto ponašanje normalno tokom terapie Spalmotila. Pre upotrebe bilo kog leka obavezno je da se posaveti sa lekarim ili farmaceutom.',
 [Document(id='b99108bc-8874-4a93-86d4-7ba1f6472116', metadata={'Naslov_1': '**<u>Posetite Vašeg lekara ukoliko Vam se tokom terapije javi neko od neželjenih dejstava u nastavku navedenih.</u>**', 'izvor_fajl': '001977657 2024 59010 007 000 515 021 04 001.pdf'}, page_content='Posetite Vašeg lekara ukoliko Vam se tokom terapije javi neko od neželjenih dejstava u nastavku navedenih.\nMože biti potrebno da se obratite Vašem lekaru ili da odmah odete u bolnicu.'),
  Document(id='5f53792f-2918-44a6-bce7-f9a957be7a63', metadata={'Naslov_1': '**Ne prekidajte terapiju lekom Spalmotil osim ukoliko Vam nije rečeno da tako učinite.**', 'izvor_fajl': '000457647 2023 59010 007 000 515 021 04 001.pdf'}, page_content='Ne prekidajte terapiju lekom Spalmotil osim ukoliko Vam nije rečeno da tako učinite.\nObavestite V

In [36]:
pitaj("Dobro. Hvala ti. A koliko je antideprisivima potrebno da deluju?")

Q: Dobro. Hvala ti. A koliko je antideprisivima potrebno da deluju?
A: 

Antidepresivi su lekovi koji se koriste za liječenje depresije, anksiositeta i drugih mentalnih zdravljanskih stanja. Vreme koje je potrebno za njihovo delovanje varira u zavisnosti od vrste leka, doze i pojedinca.

Neki antidepresivi počinju da deluju u roku od nekoliko sedmica, dok drugi mogu potrebno vreme da počnu da pokazuju svoje efekte. Generalno, najefikasniji antidepresivni lekovi su oni koji sadrže sertraline, paroksetin i venlafaksin. Oni mogu početi da deluju veoma brzo, često u roku od 1-2 nedelje.

Međutim, neke osobe mogu potrebiti više vremena da počnu doživljavati efekte antidepresiva. Neki antidepressivi, kao što su fluoksetin, sertralin i venlafsaksin, mogu potencijalno da potrebno imaju nekoliko meseci da počnu delovati.

Važno je napomenuti da svaki pojedinac reaguje drugačije na antidepresive, pa je potrebna pacijentima da budu pacijenti i da budu pod stalnim nadzorom svog lekarstva tokom ter

('\n\nAntidepresivi su lekovi koji se koriste za liječenje depresije, anksiositeta i drugih mentalnih zdravljanskih stanja. Vreme koje je potrebno za njihovo delovanje varira u zavisnosti od vrste leka, doze i pojedinca.\n\nNeki antidepresivi počinju da deluju u roku od nekoliko sedmica, dok drugi mogu potrebno vreme da počnu da pokazuju svoje efekte. Generalno, najefikasniji antidepresivni lekovi su oni koji sadrže sertraline, paroksetin i venlafaksin. Oni mogu početi da deluju veoma brzo, često u roku od 1-2 nedelje.\n\nMeđutim, neke osobe mogu potrebiti više vremena da počnu doživljavati efekte antidepresiva. Neki antidepressivi, kao što su fluoksetin, sertralin i venlafsaksin, mogu potencijalno da potrebno imaju nekoliko meseci da počnu delovati.\n\nVažno je napomenuti da svaki pojedinac reaguje drugačije na antidepresive, pa je potrebna pacijentima da budu pacijenti i da budu pod stalnim nadzorom svog lekarstva tokom terapije.',
 [Document(id='6927000f-a4b1-4f38-8c4b-2b8ded15db71'

In [37]:
pitaj("Dobro. Hvala ti na pomoći")

Q: Dobro. Hvala ti na pomoći
A: 
[12.8s]



('',
 [Document(id='37155948-d356-447b-9a70-50f0bd3af3aa', metadata={'izvor_fajl': '515-01-02931-21-001.pdf'}, page_content='**<u>UPUTSTVO ZA LEK</u>**'),
  Document(id='b434c452-06b9-4f36-9a16-3363ad1c7706', metadata={'izvor_fajl': '515-01-00448-23-001.pdf'}, page_content='**<u>UPUTSTVO ZA LEK</u>**'),
  Document(id='264ac4b0-e50d-471f-84f3-90c21441b50a', metadata={'izvor_fajl': '515-01-01364-21-001.pdf'}, page_content='**<u>UPUTSTVO ZA LEK</u>**'),
  Document(id='fdeb18fe-6ec7-4d3a-8829-85ab6661d448', metadata={'izvor_fajl': '000457379 2023 59010 007 000 515 021 04 001.pdf'}, page_content='**<u>UPUTSTVO ZA LEK</u>**'),
  Document(id='9600c7c3-8a2d-4dcb-94fb-4818babcdeec', metadata={'izvor_fajl': '000457553 2023 59010 007 000 515 021 04 001.pdf'}, page_content='**<u>UPUTSTVO ZA LEK</u>**')],
 12.84803318977356)

#### Session B continued: out-of-domain and English questions

The original inputs and generated responses are reproduced unchanged.

In [38]:
pitaj("Kaži mi šta je to okean?")

Q: Kaži mi šta je to okean?
A: 

Okean je veliki ocean koji pokriva oko 71% površine Zemlje.
[78.2s]



('\n\nOkean je veliki ocean koji pokriva oko 71% površine Zemlje.',
 [Document(id='37155948-d356-447b-9a70-50f0bd3af3aa', metadata={'izvor_fajl': '515-01-02931-21-001.pdf'}, page_content='**<u>UPUTSTVO ZA LEK</u>**'),
  Document(id='b434c452-06b9-4f36-9a16-3363ad1c7706', metadata={'izvor_fajl': '515-01-00448-23-001.pdf'}, page_content='**<u>UPUTSTVO ZA LEK</u>**'),
  Document(id='264ac4b0-e50d-471f-84f3-90c21441b50a', metadata={'izvor_fajl': '515-01-01364-21-001.pdf'}, page_content='**<u>UPUTSTVO ZA LEK</u>**'),
  Document(id='fdeb18fe-6ec7-4d3a-8829-85ab6661d448', metadata={'izvor_fajl': '000457379 2023 59010 007 000 515 021 04 001.pdf'}, page_content='**<u>UPUTSTVO ZA LEK</u>**'),
  Document(id='9600c7c3-8a2d-4dcb-94fb-4818babcdeec', metadata={'izvor_fajl': '000457553 2023 59010 007 000 515 021 04 001.pdf'}, page_content='**<u>UPUTSTVO ZA LEK</u>**')],
 78.16466975212097)

In [39]:
pitaj("Do you have an ability to talk in English?")

Q: Do you have an ability to talk in English?
A: 

I can understand and respond in English, but I'm a Serbian language model, so my primary language is Serbian. If you'd like, we can communicate in English, and I'll do my best to provide accurate and helpful responses.
[15.7s]



("\n\nI can understand and respond in English, but I'm a Serbian language model, so my primary language is Serbian. If you'd like, we can communicate in English, and I'll do my best to provide accurate and helpful responses.",
 [Document(id='ddcbd11b-8c00-4fab-95d1-c1a92452caf6', metadata={'izvor_fajl': '515-01-03270-20-001.pdf'}, page_content='**<u>UPUTSTVO ZA LEK</u>**'),
  Document(id='5ad33434-2944-4b6e-88f1-498709ea151f', metadata={'izvor_fajl': '000456975 2023 59010 007 000 515 021 04 001.pdf'}, page_content='**<u>UPUTSTVO ZA LEK</u>**'),
  Document(id='392291c3-e1f5-4521-8136-87f539eb0be5', metadata={'izvor_fajl': '515-01-04215-19-001.pdf'}, page_content='**<u>UPUTSTVO ZA LEK</u>**'),
  Document(id='14e7b4c5-4ae5-4ebb-a0a3-e87124146b02', metadata={'izvor_fajl': '515-01-00434-19-001.pdf'}, page_content='**<u>UPUTSTVO ZA LEK</u>**'),
  Document(id='a4df9157-2418-4062-a3c9-faecf5247b32', metadata={'izvor_fajl': '000457360 2023 59010 007 000 515 021 04 001.pdf'}, page_content='**<u>

In [40]:
pitaj("Can I drive a car if I take the medicin  e called Paracetamol?")

Q: Can I drive a car if I take the medicin  e called Paracetamol?
A: 

Nažalosto, nemam informaciju o tome da li je Paracetamolum AbelaPharma dozvoljeno vožnju automobila. Pre upotrebe bilo kog leka, obavezni ste da se posavete sa lekarjem ili farmaceutonom.
[33.5s]



('\n\nNažalosto, nemam informaciju o tome da li je Paracetamolum AbelaPharma dozvoljeno vožnju automobila. Pre upotrebe bilo kog leka, obavezni ste da se posavete sa lekarjem ili farmaceutonom.',
 [Document(id='d402706c-c03e-47d6-9ac0-71354423f7c1', metadata={'Naslov_1': '**Lek je namenjen za intravensku primenu Rastvor paracetamola se primenjuje kao intravenska infuzija u trajanju od 15 minuta**', 'izvor_fajl': '000461289 2023 59010 007 000 515 020 04 001.pdf'}, page_content='Lek je namenjen za intravensku primenu Rastvor paracetamola se primenjuje kao intravenska infuzija u trajanju od 15 minuta\nObratite pažnju kod propisivanja i primene leka Paracetamol AbelaPharm, rastvor za infuziju, na mogućnost zamene miligrama (mg) i mililitara (mL), kako biste izbegli greške u doziranju koje bi mogle dovesti do slučajnog predoziranja i smrti. Pripazite na to da se odgovarajuća doza navede u komunikaciji i primeni. Prilikom propisivanja recepata, navedite obe doze, na oba načina , tj. ukupnu d

## Test 2 ? Fine-tuned model without RAG

Run this section from the first cell. The test only changes the selected chain; the model, prompt and decoding settings are shared with the other section. Four isolated questions use empty history individually. Generated answers, retrieved text and metrics are saved as CSV files on Drive, overwriting earlier files for this section when rerun. Save the notebook to retain conversation outputs too.


In [41]:
koristi_rag = False
test_name = 'test_2_finetuned_no_rag'
chat_history = []


In [42]:
# Same four independent questions in all four tests.
generisani_odgovori = []
redovi = []
for primer in eval_dataset_odgovori:
    chat_history = []
    odgovor, dokumenti, trajanje = pitaj(primer['pitanje'])
    generisani_odgovori.append(odgovor)
    redovi.append({
        'Pitanje': primer['pitanje'],
        'Odgovor': odgovor,
        'Referentni odgovor': primer['referentni_odgovor'],
        'Vreme (s)': trajanje,
        'Izvori': '; '.join(str(d.metadata.get('izvor_fajl', '')) for d in dokumenti),
        'Kontekst': '\n\n'.join(d.page_content for d in dokumenti),
    })
# Save answers before calculating metrics, so a metric error does not lose them.
df_odgovori = pd.DataFrame(redovi)
df_odgovori.to_csv(output_dir / f'{test_name}_answers.csv', index=False)
display(df_odgovori[['Pitanje', 'Odgovor', 'Vreme (s)', 'Izvori']])


Q: Kada ne smem da uzimam lek Dovicin?
A: 

Ne mogu da vam pružim ličnu medicinsku savetu. Ako imate pitanja o lekovima ili zdravlju, obratite se lekaru ili farmaceutu.
[7.8s]

Q: Koja je uobičajena doza za lek Nimulid?
A: 

Nimulid je lek koji se koristi za liječenje bolova i upala. Uobičajene doze variraju u zavisnosti od vrste bolesti i uzroka bola. Generalno, doza Nimulida može varirati od 100 do 400 mg dva puta dnevno, ali ovo treba da bude određeno od strane lekara.

Pre upotrebu svakog leka, obavezni ste da se posavete sa lekarima ili farmaceutom.
[18.5s]

Q: Da li deca smeju da konzumiraju Pressing?
A: 

Ne, deca ne smeju da koriste Pressing. Pressing sadrži kofein koji može da utiče na razvoj mozga i nervnog sistema kod dece.
[6.7s]

Q: Za šta se primenjuje lek Zoloft
A: 

Zoloft (sertralin) je lek koji se koristi za lečenje depresije, anksioznosti, panike, fobije, premenstrualnog distresnog sindroma (PMS), posttraumatske stresne poremećaja (PTSD) i drugih stanja.

Nedavno je 

,Pitanje,Odgovor,Vreme (s),Izvori
0,Kada ne smem da uzimam lek Dovicin?,\n\nNe mogu da vam pružim ličnu medicinsku sav...,7.806262,
1,Koja je uobičajena doza za lek Nimulid?,\n\nNimulid je lek koji se koristi za liječenj...,18.477380,
2,Da li deca smeju da konzumiraju Pressing?,"\n\nNe, deca ne smeju da koriste Pressing. Pre...",6.711186,
3,Za šta se primenjuje lek Zoloft,\n\nZoloft (sertralin) je lek koji se koristi ...,18.018634,


In [43]:
referentni_odgovori = [p['referentni_odgovor'] for p in eval_dataset_odgovori]
bert_p, bert_r, bert_f1 = bert_score_fn(
    generisani_odgovori, referentni_odgovori,
    lang='sr', device='cpu', batch_size=1, verbose=False,
)
rezultati_odgovora = []
for i, primer in enumerate(eval_dataset_odgovori):
    gen = generisani_odgovori[i]
    ref = primer['referentni_odgovor']
    rezultati_odgovora.append({
        'Pitanje': primer['pitanje'],
        'ROUGE-L': racunaj_rouge_l(gen, ref),
        'BLEU': racunaj_bleu(gen, ref),
        'BERTScore (F1)': bert_f1[i].item(),
        'Cosine similarity': racunaj_cos_similarity(gen, ref),
        'Vreme (s)': redovi[i]['Vreme (s)'],
    })
df_metrike = pd.DataFrame(rezultati_odgovora)
display(df_metrike)
display(df_metrike.drop(columns='Pitanje').mean().to_frame('PROSEK'))
df_metrike.to_csv(output_dir / f'{test_name}_metrics.csv', index=False)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


,Pitanje,ROUGE-L,BLEU,BERTScore (F1),Cosine similarity,Vreme (s)
0,Kada ne smem da uzimam lek Dovicin?,0.062500,0.001040,0.617260,0.487603,7.806262
1,Koja je uobičajena doza za lek Nimulid?,0.183673,0.023101,0.714567,0.752694,18.477380
2,Da li deca smeju da konzumiraju Pressing?,0.108108,0.008687,0.677939,0.714706,6.711186
3,Za šta se primenjuje lek Zoloft,0.189474,0.005657,0.695182,0.834794,18.018634


,PROSEK
ROUGE-L,0.135939
BLEU,0.009622
BERTScore (F1),0.676237
Cosine similarity,0.697449
Vreme (s),12.753365


### Optional ? original conversation tests from notebook 05

These calls and the two history resets are copied in the original order. They are separate from the four isolated questions above. Run the following cells only if you also want to inspect the original multi-turn examples; they do not contribute to the metric table.


##### Session A: Dovicin in Latin script

**History reset:** the next cell starts a new conversation.

In [44]:
chat_history = []


In [45]:
pitaj("Kada ne smem da uzimam lek Dovicin?")

Q: Kada ne smem da uzimam lek Dovicin?
A: 

Ne mogu da vam pružim ličnu medicinsku savetu. Ako imate pitanja o lekovima ili zdravlju, obratite se lekaru ili farmaceutu.
[7.9s]



('\n\nNe mogu da vam pružim ličnu medicinsku savetu. Ako imate pitanja o lekovima ili zdravlju, obratite se lekaru ili farmaceutu.',
 [],
 7.869319915771484)

In [46]:
pitaj("A šta ako uzimam i drugi lek pored Dovicina? Šta da radim?")

Q: A šta ako uzimam i drugi lek pored Dovicina? Šta da radim?
A: 

Da biste preduzele sve neophodne mere, morate da razgovarate sa lekaricom ili farmaceutom o svim lekovima koje uzimate.
[6.3s]



('\n\nDa biste preduzele sve neophodne mere, morate da razgovarate sa lekaricom ili farmaceutom o svim lekovima koje uzimate.',
 [],
 6.326293468475342)

#### Session A continued: Dovicin in Cyrillic

The original notebook does not reset history here. Earlier turns remain available.

In [47]:
pitaj("Када не смем да узимам лек Dovicin?")

Q: Када не смем да узимам лек Dovicin?
A: 

Dovicin je lek koji se koristi za liječenje i sprečavanje infekcija uzrokovanih bakterijama. Međutim, postoji nekoliko situacija kada treba da se preporučuje da se ostanete bez Dovicina ili da ga uzimate sa oprezom.

Neka od ovih situacija uključuju:

- Ako imate alergiju na Dovicin ili bilo koju od njegovih sastojaka.
- Ako ste trudne ili planirate da postanete trudne.
- Aко imате bolesti jetre ili bubreža.
- Aо је имао problem sa probavom ili je imao dijagnozu ovisnosti o alkoholu.
- Aко imate povredu mozga ili moždanih vena.
- Aсо јe imao problem sa krvnim zgrušavanjem ili je imalo anemiju.
- A се liječi sa drugim lekovima koji sadrže cefalosporine ili karbapeneme.

U svakom slučaju, prema preporuci, morate da se obratite svom lekaru ili farmaceutu da bi se osigrali da je Dovicin bezbedan za vas da uzimate.
[47.8s]



('\n\nDovicin je lek koji se koristi za liječenje i sprečavanje infekcija uzrokovanih bakterijama. Međutim, postoji nekoliko situacija kada treba da se preporučuje da se ostanete bez Dovicina ili da ga uzimate sa oprezom.\n\nNeka od ovih situacija uključuju:\n\n- Ako imate alergiju na Dovicin ili bilo koju od njegovih sastojaka.\n- Ako ste trudne ili planirate da postanete trudne.\n- Aко imате bolesti jetre ili bubreža.\n- Aо је имао problem sa probavom ili je imao dijagnozu ovisnosti o alkoholu.\n- Aко imate povredu mozga ili moždanih vena.\n- Aсо јe imao problem sa krvnim zgrušavanjem ili je imalo anemiju.\n- A се liječi sa drugim lekovima koji sadrže cefalosporine ili karbapeneme.\n\nU svakom slučaju, prema preporuci, morate da se obratite svom lekaru ili farmaceutu da bi se osigrali da je Dovicin bezbedan za vas da uzimate.',
 [],
 47.84545373916626)

In [48]:
pitaj("Зашто си ми генерисао одговор на латиници?")

Q: Зашто си ми генерисао одговор на латиници?
A: 
Молим за извинујење, али сам генерирао одговор у латиницом јер сам програм за вештачку интелигенцију обучен да користи латиничан алфавит за писање. Међутим, могу да променим своје одговоре да буду написани на ćирилици уколико вам то одговара.
[16.7s]



('\nМолим за извинујење, али сам генерирао одговор у латиницом јер сам програм за вештачку интелигенцију обучен да користи латиничан алфавит за писање. Међутим, могу да променим своје одговоре да буду написани на ćирилици уколико вам то одговара.',
 [],
 16.672109603881836)

In [49]:
pitaj("Али већ си генерисао претходни одговор на српском. Генериши одговор на ћирилици.")

Q: Али већ си генерисао претходни одговор на српском. Генериши одговор на ћирилици.
A: 
Молим да ми објасните шта желите да питам.
[3.0s]



('\nМолим да ми објасните шта желите да питам.', [], 3.009545087814331)

#### Session A continued: Nimulid

This block continues the same history; it is not an isolated test.

In [50]:
pitaj("Која је уобичајена доза за лек Nimulid?")

Q: Која је уобичајена доза за лек Nimulid?
A: 

Уобичајене дозе Нимулида варирају у зависности од узрока лечења и старости пацијента. Обично се препоручује да се узима 100-200 мг два пута дневно, али може се повећати до 400 мг два путу дневно у случајевима тешких болести. Међутим, сваки пацијент треба да прати упутства свог лекара или фармацеута пре него што почиње да узима Нимулид.
[21.8s]



('\n\nУобичајене дозе Нимулида варирају у зависности од узрока лечења и старости пацијента. Обично се препоручује да се узима 100-200 мг два пута дневно, али може се повећати до 400 мг два путу дневно у случајевима тешких болести. Међутим, сваки пацијент треба да прати упутства свог лекара или фармацеута пре него што почиње да узима Нимулид.',
 [],
 21.82824945449829)

In [51]:
pitaj("Мислим да сам случајно прекорачио дозу. Узео сам по једну таблету више сваки пут. Шта ми се може десити?")

Q: Мислим да сам случајно прекорачио дозу. Узео сам по једну таблету више сваки пут. Шта ми се може десити?
A: 

Најбоље је да се обратите својем лекара или фармацеуту за савет.
[5.1s]



('\n\nНајбоље је да се обратите својем лекара или фармацеуту за савет.',
 [],
 5.1106276512146)

In [52]:
pitaj("Такође сам конзумирао алкохол. Да ли ми се нешто може десити?")

Q: Такође сам конзумирао алкохол. Да ли ми се нешто може десити?
A: 

Алкохол је потенцијално штетан за здравље и може изазвати различите проблеме, посебно када се конзумира у великим количинама или редовно. Међутим, без обзира на то да ли сте конзументовали алкохолну пиће или не, постоје неки ризици који треба узети у обзир.

Неки од могућих проблема који могу настати након конзумовања алкохола укључују:

- Алкохолна интоксикација: Ако се алкохолови нивои у крви повећају, може доћи до алкохольне интоксикации, што може изазваћи симптоме као што су глуве, визије, дискоординација, губитак свести, повећана температура тела, повеђење, повеќе сузења, повељењем, повењеност, повеџењу, повејењ, повеежењ и повееженост, а понекад и повећани ризик од несреће.

- Оштећењи органа: Дугорочни конзумеризам алкохолица може довести до оштећених органа као шо што је жива мембрана, печењак, плућа, срце, бубрег, мозак, јетра, желудац, слезина, плуц, плуцево срце и плуцни срце.

- Психолошки ефекти: Алкохол

('\n\nАлкохол је потенцијално штетан за здравље и може изазвати различите проблеме, посебно када се конзумира у великим количинама или редовно. Међутим, без обзира на то да ли сте конзументовали алкохолну пиће или не, постоје неки ризици који треба узети у обзир.\n\nНеки од могућих проблема који могу настати након конзумовања алкохола укључују:\n\n- Алкохолна интоксикација: Ако се алкохолови нивои у крви повећају, може доћи до алкохольне интоксикации, што може изазваћи симптоме као што су глуве, визије, дискоординација, губитак свести, повећана температура тела, повеђење, повеќе сузења, повељењем, повењеност, повеџењу, повејењ, повеежењ и повееженост, а понекад и повећани ризик од несреће.\n\n- Оштећењи органа: Дугорочни конзумеризам алкохолица може довести до оштећених органа као шо што је жива мембрана, печењак, плућа, срце, бубрег, мозак, јетра, желудац, слезина, плуц, плуцево срце и плуцни срце.\n\n- Психолошки ефекти: Алкохолне употребе могу изазвати психолошке ефекте као сто су а

##### Session B: Pressing

**History reset:** the next cell starts a new conversation.

In [53]:
chat_history = []


In [54]:
pitaj("Da li deca smeju da konzumiraju Pressing?")

Q: Da li deca smeju da konzumiraju Pressing?
A: 

Ne, deca ne smeju da koriste Pressing. Pressing sadrži kofein koji može da utiče na razvoj mozga i nervnog sistema kod dece.
[6.7s]



('\n\nNe, deca ne smeju da koriste Pressing. Pressing sadrži kofein koji može da utiče na razvoj mozga i nervnog sistema kod dece.',
 [],
 6.683977842330933)

In [55]:
pitaj("Da li odrasle osobe mogu da konzumiraju Pressing?")

Q: Da li odrasle osobe mogu da konzumiraju Pressing?
A: 

Prema dostupnim informacijama, Pressing je suplement koji sadrži kombinaciju vitamina i minerala, a ne lek. Prema proizvođačima, Pressing može biti koristan za odraslu populaciju, ali je važno da se konzumira u dozama preporučene na etiketi.

Prema Nacionalnom centru za zdravlje i bolesti (NCBI), ne postoje dokazi da bi Pressing mogao da ima značajne efekte na zdravljenje odraslih osoba. Međutim, neki od sastojaka u Pressingu, kao što su vitamin D i kalcijum, mogu bitno da utiču na zdravstven status odraslog čoveka.

Prema American Academy of Family Physicians (AAFP), odraslim osobama treba da se daju doze vitamina D i kalsija koje su preporučile Američka akademija pedijatrije. Ako ste zainteresovani da li ste u potpunosti zadovoljavate ovim preporukama, savetujte se sa lekaricom ili farmaceutom.

Sve u svemu, Pressing se smatra bezbednim za odraslje ako se konzumuje u dozama koje su navedene na etiku. Međutinme, nikada ne zabor

('\n\nPrema dostupnim informacijama, Pressing je suplement koji sadrži kombinaciju vitamina i minerala, a ne lek. Prema proizvođačima, Pressing može biti koristan za odraslu populaciju, ali je važno da se konzumira u dozama preporučene na etiketi.\n\nPrema Nacionalnom centru za zdravlje i bolesti (NCBI), ne postoje dokazi da bi Pressing mogao da ima značajne efekte na zdravljenje odraslih osoba. Međutim, neki od sastojaka u Pressingu, kao što su vitamin D i kalcijum, mogu bitno da utiču na zdravstven status odraslog čoveka.\n\nPrema American Academy of Family Physicians (AAFP), odraslim osobama treba da se daju doze vitamina D i kalsija koje su preporučile Američka akademija pedijatrije. Ako ste zainteresovani da li ste u potpunosti zadovoljavate ovim preporukama, savetujte se sa lekaricom ili farmaceutom.\n\nSve u svemu, Pressing se smatra bezbednim za odraslje ako se konzumuje u dozama koje su navedene na etiku. Međutinme, nikada ne zaboravite da se pre upotrebe bilo kog suplementa s

In [56]:
pitaj("Ali ja te ne pitam za lek Crestora, već za lek Pressing?")

Q: Ali ja te ne pitam za lek Crestora, već za lek Pressing?
A: 

Ne mogu da pružim medicinske savete ili preporuke. Ako imate pitanja o leku, najbolji bi bili da se obratite lekaru ili farmaceutu.
[7.6s]



('\n\nNe mogu da pružim medicinske savete ili preporuke. Ako imate pitanja o leku, najbolji bi bili da se obratite lekaru ili farmaceutu.',
 [],
 7.617072820663452)

#### Session B continued: more specific wording

History is retained while the wording of the question changes.

In [57]:
pitaj("Da li deca smeju da konzumiraju Pressing sirup?")

Q: Da li deca smeju da konzumiraju Pressing sirup?
A: 

Ne, deca ne smeju da uzimaju Pressing Sirup bez prethodnog saveta lekara ili farmaceuta.
[5.1s]



('\n\nNe, deca ne smeju da uzimaju Pressing Sirup bez prethodnog saveta lekara ili farmaceuta.',
 [],
 5.069292783737183)

In [58]:
pitaj("Šta je to Pressing sirup?")

Q: Šta je to Pressing sirup?
A: 

Pressing sirup je lek koji se koristi za liječenje i sprečavanje napada astme. Sastoji se od kombinacije lekova koji su poznati kao beta-2 agonisti i antihistaminici. Beta-2 agonist smanjuje otpuštanje histamina iz ćelija, što dovodi do smanjenja simptoma astme kao što su kašalj, osip i disanje. Antihistaminicima se smanjuju simptomi kao što su osip, suvi u ustima i bolovi u grlu.
[21.5s]



('\n\nPressing sirup je lek koji se koristi za liječenje i sprečavanje napada astme. Sastoji se od kombinacije lekova koji su poznati kao beta-2 agonisti i antihistaminici. Beta-2 agonist smanjuje otpuštanje histamina iz ćelija, što dovodi do smanjenja simptoma astme kao što su kašalj, osip i disanje. Antihistaminicima se smanjuju simptomi kao što su osip, suvi u ustima i bolovi u grlu.',
 [],
 21.45478343963623)

In [59]:
pitaj("Da li deca smeju da konzumiraju Pressing sirup?")

Q: Da li deca smeju da konzumiraju Pressing sirup?
A: 

Ne, deca ne smeju da uzimaju Pressing Sirup bez prethodnog saveta lekara ili farmaceuta.
[5.1s]



('\n\nNe, deca ne smeju da uzimaju Pressing Sirup bez prethodnog saveta lekara ili farmaceuta.',
 [],
 5.0754475593566895)

In [60]:
pitaj("A deca starija od 2 godine?")

Q: A deca starija od 2 godine?
A: 

Deca stariji od dve godine mogu da uzimaju različite vrste tableta i kapsula, ali to zavisi od njihove dobre volje i sposobnosti da ih popiju. Neka se uzme u obzir da deca stara 2 godina moraju da imaju dovoljno dobrovoljnih pokreta u žvakaću da bi mogli da popiju tablete ili kapsule.

U slučaju da deca ne mogu da popiju tabletu ili kapsulu, postoje i druge načine da daju lekove, kao što su:

- Sirupovi: Sirupovi su idealni za deca koja nemaju sposobnost da popiju pilule ili kapsule. Oni su ukusni i laki za popijanje, a dostupni su u raznim ukusima.
- Kapsule koje se otvaraju: Kapsule se mogu otvoriti i da se napune sirupom ili drugim oblikom leka koji je lakši za popijenje.
- Lekovi u obliku čvrstih tableta: Dečiji tableti su dizajnirani da budu lakši za pojanje i mogu bit da sadrže manje količine aktivnih sastojaka od odraslih tableta.

Sve u svemu, važno je da se pre upotrebe bilo kog leka savetujete sa lekaricom ili farmaceutom kako biste obezbed

('\n\nDeca stariji od dve godine mogu da uzimaju različite vrste tableta i kapsula, ali to zavisi od njihove dobre volje i sposobnosti da ih popiju. Neka se uzme u obzir da deca stara 2 godina moraju da imaju dovoljno dobrovoljnih pokreta u žvakaću da bi mogli da popiju tablete ili kapsule.\n\nU slučaju da deca ne mogu da popiju tabletu ili kapsulu, postoje i druge načine da daju lekove, kao što su:\n\n- Sirupovi: Sirupovi su idealni za deca koja nemaju sposobnost da popiju pilule ili kapsule. Oni su ukusni i laki za popijanje, a dostupni su u raznim ukusima.\n- Kapsule koje se otvaraju: Kapsule se mogu otvoriti i da se napune sirupom ili drugim oblikom leka koji je lakši za popijenje.\n- Lekovi u obliku čvrstih tableta: Dečiji tableti su dizajnirani da budu lakši za pojanje i mogu bit da sadrže manje količine aktivnih sastojaka od odraslih tableta.\n\nSve u svemu, važno je da se pre upotrebe bilo kog leka savetujete sa lekaricom ili farmaceutom kako biste obezbedili da je lek siguran 

#### Session B continued: repeated standalone wording

Although the question is self-contained, the original code does not reset conversation history.

In [61]:
pitaj("Šta je to Pressing sirup?")

Q: Šta je to Pressing sirup?
A: 

Pressing sirup je lek koji se koristi za liječenje i sprečavanje napada astme. Sastoji se od kombinacije lekova koji su poznati kao beta-2 agonisti i antihistaminici. Beta-2 agonist smanjuje otpuštanje histamina iz ćelija, što dovodi do smanjenja simptoma astme kao što su kašalj, osip i disanje. Antihistaminicima se smanjuju simptomi kao što su osip, suvi u ustima i bolovi u grlu.
[21.3s]



('\n\nPressing sirup je lek koji se koristi za liječenje i sprečavanje napada astme. Sastoji se od kombinacije lekova koji su poznati kao beta-2 agonisti i antihistaminici. Beta-2 agonist smanjuje otpuštanje histamina iz ćelija, što dovodi do smanjenja simptoma astme kao što su kašalj, osip i disanje. Antihistaminicima se smanjuju simptomi kao što su osip, suvi u ustima i bolovi u grlu.',
 [],
 21.31171727180481)

#### Session B continued: Zoloft

The existing history remains in use.

In [62]:
pitaj("Za šta se primenjuje lek Zoloft")

Q: Za šta se primenjuje lek Zoloft
A: 

Zoloft (sertralin) je lek koji se koristi za lečenje depresije, anksioznosti, panike, fobije, premenstrualnog distresnog sindroma (PMS), posttraumatske stresne poremećaja (PTSD) i drugih stanja.

Nedavno je FDA odobrio Zoloft za lečenju osećanja gladi za hranom kod osoba sa anoreksijom nervozom.
[18.0s]



('\n\nZoloft (sertralin) je lek koji se koristi za lečenje depresije, anksioznosti, panike, fobije, premenstrualnog distresnog sindroma (PMS), posttraumatske stresne poremećaja (PTSD) i drugih stanja.\n\nNedavno je FDA odobrio Zoloft za lečenju osećanja gladi za hranom kod osoba sa anoreksijom nervozom.',
 [],
 18.03027081489563)

In [63]:
pitaj("Započeo sam terapiju ovim lekom, da li je normalno što ponekad pomislim o samopovređivanju?")

Q: Započeo sam terapiju ovim lekom, da li je normalno što ponekad pomislim o samopovređivanju?
A: 

Ne mogu da dajem medicinske savete.
[2.5s]



('\n\nNe mogu da dajem medicinske savete.', [], 2.495790958404541)

In [64]:
pitaj("Dobro. Hvala ti. A koliko je antideprisivima potrebno da deluju?")

Q: Dobro. Hvala ti. A koliko je antideprisivima potrebno da deluju?
A: 

Antidepresivi su lekovi koji se koriste za lečenje depresije, anksioznosti i drugih mentalnih zdravstvenih stanja. Vreme koje je potrebno za početak dejstva varira od osobe do osobe i od vrste leka.

Uopšteno, većina antidepresiva počinje da deluje nakon nekoliko nedelja, a najefikasniji rezultati se obično postižu nakon 6-8 nedelja. Međutim, neke osobe mogu da vide poboljšanje već nakon nekih nedelja.

Neka antidepresna leka počinju da deluju brže, dok drugi mogu da treba nekoliko meseci da bi postigli svoj maksimalan efekat. Važno je da budete pacijent i da nastavite da uzimate lek kako je preporučeno od strane vašeg lekara ili farmaceuta.

Sve u svemu, važno je napomenuti da svaki čovek reaguje drugačije na različite lekove, pa je važno da nastavite sa ličnim pregledom sa vašim lekarom kako bi se utvrdilo da li je lek koji uzimate efikasan i da li je potrebna promena.
[47.5s]



('\n\nAntidepresivi su lekovi koji se koriste za lečenje depresije, anksioznosti i drugih mentalnih zdravstvenih stanja. Vreme koje je potrebno za početak dejstva varira od osobe do osobe i od vrste leka.\n\nUopšteno, većina antidepresiva počinje da deluje nakon nekoliko nedelja, a najefikasniji rezultati se obično postižu nakon 6-8 nedelja. Međutim, neke osobe mogu da vide poboljšanje već nakon nekih nedelja.\n\nNeka antidepresna leka počinju da deluju brže, dok drugi mogu da treba nekoliko meseci da bi postigli svoj maksimalan efekat. Važno je da budete pacijent i da nastavite da uzimate lek kako je preporučeno od strane vašeg lekara ili farmaceuta.\n\nSve u svemu, važno je napomenuti da svaki čovek reaguje drugačije na različite lekove, pa je važno da nastavite sa ličnim pregledom sa vašim lekarom kako bi se utvrdilo da li je lek koji uzimate efikasan i da li je potrebna promena.',
 [],
 47.50976777076721)

In [65]:
pitaj("Dobro. Hvala ti na pomoći")

Q: Dobro. Hvala ti na pomoći
A: 

Srećan sam što sam mogao da vam pomognem. Ako imate bilo kakva druga pitanja ili potrebujete dalju pomoć, ne sumnjajte da me pitajte.
[8.1s]



('\n\nSrećan sam što sam mogao da vam pomognem. Ako imate bilo kakva druga pitanja ili potrebujete dalju pomoć, ne sumnjajte da me pitajte.',
 [],
 8.132033824920654)

#### Session B continued: out-of-domain and English questions

The original inputs and generated responses are reproduced unchanged.

In [66]:
pitaj("Kaži mi šta je to okean?")

Q: Kaži mi šta je to okean?
A: 

Okean je veliki ocean koji pokriva oko 71% površine Zemlje.
[4.4s]



('\n\nOkean je veliki ocean koji pokriva oko 71% površine Zemlje.',
 [],
 4.393737077713013)

In [67]:
pitaj("Do you have an ability to talk in English?")

Q: Do you have an ability to talk in English?
A: 

Da, razumem engleski.
[1.8s]



('\n\nDa, razumem engleski.', [], 1.7985546588897705)

In [68]:
pitaj("Can I drive a car if I take the medicin  e called Paracetamol?")

Q: Can I drive a car if I take the medicin  e called Paracetamol?
A: 

Da li možete vožnju vozila dok uzimate paracetamol?

Paracetamol je bezbedan lek koji se može uzeti bez pretnje da će uticati na vašu sposobnost vožnje. Međutim, ako ste uzeli paracetamolo u kombinaciji sa drugim lekovima ili imate neke prethodne zdravstvene stanje, treba da se obratite svom lekaru ili farmaceutu da bi se potvrdilo da je sigurno da možete voznju vozila.

Prema Nacionalnom centru za zdravstvo i zdravku (NHS), paracetamola je generalno bezbedna za vožnja vozila, ali ako ste uzimali više od preporučene doze ili imate druge zdravštenske stanje, to može uticati vašoj sposobnosti vožnji vozila.

Sve u svemu, najbolji je da se obrate svom lekara ili farmaceuti da bi se obezbedio da je siguran da možete da voznju voza dok uzimate paracetamole.
[42.1s]



('\n\nDa li možete vožnju vozila dok uzimate paracetamol?\n\nParacetamol je bezbedan lek koji se može uzeti bez pretnje da će uticati na vašu sposobnost vožnje. Međutim, ako ste uzeli paracetamolo u kombinaciji sa drugim lekovima ili imate neke prethodne zdravstvene stanje, treba da se obratite svom lekaru ili farmaceutu da bi se potvrdilo da je sigurno da možete voznju vozila.\n\nPrema Nacionalnom centru za zdravstvo i zdravku (NHS), paracetamola je generalno bezbedna za vožnja vozila, ali ako ste uzimali više od preporučene doze ili imate druge zdravštenske stanje, to može uticati vašoj sposobnosti vožnji vozila.\n\nSve u svemu, najbolji je da se obrate svom lekara ili farmaceuti da bi se obezbedio da je siguran da možete da voznju voza dok uzimate paracetamole.',
 [],
 42.13532876968384)

## Comparing the four tests

- **Test 4 versus test 1:** base + RAG versus fine-tuned + RAG, using the same isolated questions and retrieval setup. This is the main fine-tuning comparison.
- **Test 3 versus test 2:** base versus fine-tuned with empty context; evaluate appropriate refusal as well as wording.
- **Test 1 versus test 2:** context available versus absent for the fine-tuned model.
- **Test 4 versus test 3:** context available versus absent for the base model.

Use the saved answers to assess factual correctness, completeness, Serbian language, script, disclaimer, and whether a refusal was appropriate. A correct refusal without context should not be judged incorrect just because it differs from an answer-bearing reference. All four tests use 8-bit loading. There is no quantization ablation. Run the two notebooks in separate sessions or restart the runtime before switching, so only one generator occupies memory.


## Recorded follow-up results

All four conditions across notebooks 07 and 08 completed on 24 October 2026. See the [published findings and CSV exports](../results/README.md). The saved RAG contexts match across models. RAG improved all four mean similarity metrics for both models, while additional fine-tuning did not improve the means with RAG on this four-question pilot. The original prompt calls for refusal without context, so answer similarity alone cannot assess that condition. The report also records language/factual failures and the remaining Colab requests dependency conflict.
